# 06_Classification: ds002778 — Resting State EEG

**Author:** Fajar Laksono

## 0. Overview

This notebook trains and evaluates classical machine learning classifiers on the ROI band-power features produced in notebook 04 and statistically validated in notebook 05. The goal is to determine how well EEG features can distinguish Parkinson's Disease patients from healthy controls — and whether medication state (off vs on levodopa) is detectable from brain signals alone.

### 0.1. What we learned from notebooks 01–05

| Notebook | Finding relevant to classification |
|----------|------------------------------------|
| **02 EDA** | PD-off shows ~30% higher delta power vs HC; elevated beta in both PD groups |
| **03 Preprocessing** | All 46/46 sessions clean; epoch counts vary (22–103) — some subjects have very few epochs |
| **04 Features** | 45 ROI features; top Cohen's d features are `abs_theta_occipital` (d=1.19), `abs_theta_parietal` (d=1.12); PCA shows HC separates from PD but PD-off/PD-on overlap |
| **05 Statistics** | 7 theta features survive FDR; HC vs PD-off has large d (0.58–1.16) but is underpowered at n=15; HC < PD-off < PD-on in theta (monotonic) |

### 0.2. Two classification problems

| Problem | Classes | n | Key challenge |
|---------|---------|---|---------------|
| **Binary** | HC=0 vs PD=1 (pool PD-off + PD-on) | 16 vs 30 | Class imbalance (1:1.9) |
| **3-class** | HC vs PD-off vs PD-on | 16/15/15 | PD-off vs PD-on boundary (max Cohen's d = 0.52) |

Machine learning models perform best when they have an equal number of examples for each class. Here, you have nearly twice as many PD examples as HC examples. If a model is "lazy," it might just learn to guess "PD" most of the time to achieve high accuracy, which makes it less reliable at identifying healthy individuals.

Cohen’s d (0.52): This is a measure of effect size. It tells you how much the "signal" of the data differs between groups. (0.2 is a small effect, 0.5 is a medium effect, and 0.8 is a large effect). A Cohen's d of 0.52 suggests that while there is a visible difference in the data between the ON and OFF states, it is only "moderate." There is likely a lot of overlap in the data points, making it difficult for a classifier to draw a clean line between them.

### 0.3. Two feature sets

| Set | n features | Source |
|-----|-----------|--------|
| **Primary** | 7 | FDR-significant theta features from notebook 05 (`selected_features.txt`) |
| **Extended** | 45 | All ROI features (4 ROIs × 5 bands × abs+rel + 4 APF + 1 global APF) |

### 0.4. Pipeline

| Step | Section | Action |
|------|---------|--------|
| 1 | §2 | Load `features.csv` + `selected_features.txt` |
| 2 | §3 | Define subject-level LOSO cross-validation |
| 3 | §4 | Define classifiers (LDA, SVM, Random Forest) with rationale |
| 4 | §5 | Helper functions: LOSO loop, metrics |
| 5 | §6 | Binary classification (HC vs PD) |
| 6 | §7 | Three-class classification (HC/PD-off/PD-on) |
| 7 | §8 | Results comparison table |
| 8 | §9 | Visualizations: confusion matrices, ROC, feature importance, per-subject accuracy |
| 9 | §10 | Conclusions |

## 1. Preparations

### 1.1. Import Libraries

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, auc
)

warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


### 1.2. Configuration

In [2]:
PROC_DIR      = 'processed'
FEAT_FILE     = os.path.join(PROC_DIR, 'features.csv')
SEL_FEAT_FILE = os.path.join(PROC_DIR, 'selected_features.txt')

GROUPS_3     = ['HC', 'PD-off', 'PD-on']
GROUP_COLORS = {'HC': 'steelblue', 'PD-off': 'tomato', 'PD-on': 'darkorange'}

print(f'Feature file          : {os.path.abspath(FEAT_FILE)}')
print(f'Selected feature file : {os.path.abspath(SEL_FEAT_FILE)}')

Feature file          : D:\Project\Github\FajarLaksono\ai-neuro-EEG-ds002778-analysis\processed\features.csv
Selected feature file : D:\Project\Github\FajarLaksono\ai-neuro-EEG-ds002778-analysis\processed\selected_features.txt


## 2. Load Data

We load the 46-session ROI feature matrix from notebook 04 and the 7 FDR-validated theta features selected in notebook 05.

Two label columns are created:
- `label` — 3-class: HC=0, PD-off=1, PD-on=2
- `binary_label` — binary: HC=0, PD (off+on)=1

In [ ]:
df = pd.read_csv(FEAT_FILE)
all_feat_cols = [c for c in df.columns if c.startswith(('abs_', 'rel_', 'apf_'))]

with open(SEL_FEAT_FILE) as f:
    sel_feat_cols = [line.strip() for line in f if line.strip()]

# Binary label: HC=0, PD (both sessions) = 1
df['binary_label'] = df['label'].apply(lambda x: 0 if x == 0 else 1)

print(f'Dataset shape     : {df.shape}')
print(f'Subjects          : {df["subject"].nunique()} unique subjects')
print(f'All features      : {len(all_feat_cols)}')
print(f'Selected features : {len(sel_feat_cols)}')
print(f'  {sel_feat_cols}')
print()
print('Group / label breakdown:')
print(df.groupby(['group', 'binary_label', 'label']).size().rename('n_sessions').to_string())

## 3. Leave-One-Subject-Out (LOSO) Cross-Validation

### Why LOSO and not k-fold or holdout?

With only n=46 sessions from 31 subjects, we have two competing concerns:

1. **Data leakage risk**: PD subjects each contribute **two sessions** (ses-off and ses-on). If we randomly split sessions into train/test, a PD subject's ses-off could end up in training while their ses-on is in the test set — the model would be partially tested on a person it already trained on. This inflates accuracy artificially.

2. **Generalization target**: In clinical practice, a classifier must work on a *new patient* it has never seen. LOSO directly simulates this by leaving out **all sessions from one subject**, training on everyone else, and testing on the held-out subject.

### How it works

```
31 folds — one per unique subject

Fold for HC subject i  →  train: 45 sessions  |  test: 1 session  (HC)
Fold for PD subject j  →  train: 44 sessions  |  test: 2 sessions (PD-off + PD-on)
```

The **StandardScaler is fit on the training data only** inside each fold. This prevents any test-set statistics (e.g., mean or variance) from contaminating the training process — a subtle but critical form of data leakage.

In [ ]:
subjects = df['subject'].unique()
hc_subs  = sorted([s for s in subjects if 'hc' in s])
pd_subs  = sorted([s for s in subjects if 'pd' in s])

print(f'Total LOSO folds: {len(subjects)}')
print(f'  HC subjects ({len(hc_subs)}) → 1 test session per fold')
print(f'  PD subjects ({len(pd_subs)}) → 2 test sessions per fold (off + on)')
print()

# Illustrate one fold
print('Example fold — leaving out sub-pd11:')
test_ex  = df[df['subject'] == 'pd11'][['subject', 'session', 'group', 'label']]
train_ex = df[df['subject'] != 'pd11']
print(f'  TEST  ({len(test_ex)} sessions):')
print(test_ex.to_string(index=False))
print(f'  TRAIN ({len(train_ex)} sessions): {train_ex["group"].value_counts().to_dict()}')

## 4. Model Selection & Rationale

Three classifiers are used. Each was chosen for a specific reason grounded in the properties of this dataset.

---

### 4.1. Linear Discriminant Analysis (LDA)

**What it does:** LDA finds a linear combination of features that maximally separates the class means relative to within-class variance. For binary classification, it reduces the problem to a single discriminant axis.

**Why it fits this dataset:**
- Designed specifically for small-n, multi-feature scenarios — exactly our situation (n=46, 7–45 features)
- Notebook 05 showed that 80.7% of features are consistent with normality across groups, which is the key assumption LDA makes
- Linear boundaries are appropriate here: the PCA in notebook 04 showed HC and PD separate along a primary axis (PC1), suggesting a linear boundary may be sufficient
- Produces interpretable coefficients that map directly onto which brain features drive the classification

**Limitation:** Assumes equal covariance across classes. If PD-off has higher within-group variability (which it does — CV=0.51 vs HC CV=0.23 for some features), LDA may be slightly suboptimal.

**In the EEG/BCI literature:** LDA is the most widely used classifier in P300 BCI systems and is the standard baseline in all EEG classification papers.

---

### 4.2. Support Vector Machine (SVM, RBF kernel)

**What it does:** SVM finds the hyperplane that maximizes the margin between classes. The RBF (Radial Basis Function) kernel implicitly maps features into a higher-dimensional space, allowing it to capture non-linear relationships.

**Why it fits this dataset:**
- **Maximum margin principle** — with small n, overfitting is the primary risk. SVM's margin maximization provides a strong regularization effect that generalizes well at low sample sizes
- **RBF kernel** handles cases where the class boundary is not perfectly linear — e.g., some PD subjects might cluster differently from others due to disease severity
- `class_weight='balanced'` automatically adjusts for the binary class imbalance (HC=16, PD=30) by up-weighting the minority class
- Most cited classifier in EEG-based PD detection literature (Yuvaraj et al. 2016, Chaturvedi et al. 2019)

**Limitation:** RBF SVM is a black box — it does not directly tell you *which features* drove the classification. This is why we also use Random Forest for feature importance.

**In the EEG/BCI literature:** SVM with RBF kernel was the dominant method in EEG classification from ~2005–2018 and remains a strong baseline.

---

### 4.3. Random Forest (RF)

**What it does:** An ensemble of decision trees, each trained on a bootstrap sample of the data and a random subset of features. The final prediction is a majority vote across all trees.

**Why it fits this dataset:**
- **Handles correlated features naturally** — EEG band powers within the same ROI are highly correlated (e.g., abs_theta_central and rel_theta_central). Decision trees are not affected by multicollinearity the way linear models are
- **Feature importance** — RF provides Mean Decrease in Impurity (MDI) scores for each feature, which will feed directly into the interpretation in notebook 08 (SHAP)
- **Bootstrap aggregation** reduces variance — important when some subjects have very few epochs (pd6: 22, pd26: 23)
- `class_weight='balanced'` and `max_features='sqrt'` prevent the majority class from dominating and reduce overfitting

**Limitation:** With n=46, each bootstrap sample is small and individual trees are shallow. Performance may be lower than SVM for very small feature sets.

**In the EEG/BCI literature:** Increasingly used in clinical EEG research (2015–present) specifically because of feature importance for biomarker interpretation.

---

> **Note on `class_weight='balanced'`:**  
> In binary classification, HC has 16 sessions and PD has 30. Without correction, a classifier that predicts PD for everyone would reach 65% accuracy — a misleading result. `class_weight='balanced'` assigns each class a weight proportional to its inverse frequency, so the model is penalized equally for misclassifying HC as for misclassifying PD.

In [ ]:
CLASSIFIERS = {
    'LDA': LinearDiscriminantAnalysis(solver='svd'),
    'SVM': SVC(
        kernel='rbf', C=1.0, gamma='scale',
        probability=True,          # enables predict_proba via Platt scaling
        class_weight='balanced',
        random_state=42
    ),
    'RF':  RandomForestClassifier(
        n_estimators=200,
        max_features='sqrt',       # sqrt(n_features) per split — standard anti-overfit setting
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
}

print('Classifiers defined:')
for name, clf in CLASSIFIERS.items():
    print(f'  {name:4s} : {clf}')

## 5. Helper Functions

Two core functions encapsulate the LOSO loop and metric computation. Centralising these ensures that binary and 3-class experiments use exactly the same evaluation logic.

In [ ]:
def run_loso(feat_df, feature_cols, label_col, classifiers):
    """
    Subject-level Leave-One-Subject-Out cross-validation.

    For each unique subject, ALL their sessions are held out as the test set.
    The StandardScaler is fit on the training fold only — never on the full
    dataset — to prevent information from the test subject leaking into training.

    Parameters
    ----------
    feat_df      : pd.DataFrame — the full feature table (features.csv)
    feature_cols : list[str]   — column names to use as input features
    label_col    : str         — column name of the target label
    classifiers  : dict        — {name: sklearn estimator}

    Returns
    -------
    dict : clf_name -> {'y_true', 'y_pred', 'y_prob', 'subject'}
    """
    subjects = feat_df['subject'].unique()
    res = {name: {'y_true': [], 'y_pred': [], 'y_prob': [], 'subject': []}
           for name in classifiers}

    for test_sub in subjects:
        train = feat_df[feat_df['subject'] != test_sub]
        test  = feat_df[feat_df['subject'] == test_sub]

        # Fit scaler on TRAINING data only
        scaler  = StandardScaler()
        X_train = scaler.fit_transform(train[feature_cols])
        X_test  = scaler.transform(test[feature_cols])
        y_train = train[label_col].values
        y_test  = test[label_col].values

        for name, clf in classifiers.items():
            c = clone(clf)           # fresh estimator each fold
            c.fit(X_train, y_train)
            y_pred = c.predict(X_test)
            y_prob = c.predict_proba(X_test)  # shape (n_test, n_classes)

            res[name]['y_true'].extend(y_test.tolist())
            res[name]['y_pred'].extend(y_pred.tolist())
            res[name]['y_prob'].extend(y_prob.tolist())
            res[name]['subject'].extend([test_sub] * len(y_test))

    return res


def compute_metrics(res, n_classes):
    """
    Compute Accuracy, Balanced Accuracy, F1 Macro, and AUC-ROC from LOSO results.

    Balanced Accuracy is the primary metric:
    it reports the mean per-class recall, making it robust to the class imbalance
    present in the binary problem (HC=16, PD=30).
    """
    y_true = np.array(res['y_true'])
    y_pred = np.array(res['y_pred'])
    y_prob = np.array(res['y_prob'])

    m = {
        'Accuracy':      round(accuracy_score(y_true, y_pred), 3),
        'Bal. Accuracy': round(balanced_accuracy_score(y_true, y_pred), 3),
        'F1 Macro':      round(f1_score(y_true, y_pred, average='macro', zero_division=0), 3),
    }

    classes = sorted(np.unique(y_true))
    try:
        if n_classes == 2:
            m['AUC-ROC'] = round(roc_auc_score(y_true, y_prob[:, 1]), 3)
        else:
            y_bin = label_binarize(y_true, classes=classes)
            m['AUC-ROC'] = round(
                roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro'), 3)
    except Exception:
        m['AUC-ROC'] = float('nan')

    return m


def summarise(results_dict, n_classes, label=''):
    """Print and return a metrics DataFrame for a set of LOSO results."""
    rows = []
    for name, res in results_dict.items():
        m = compute_metrics(res, n_classes)
        rows.append({'Classifier': name, **m})
    df_m = pd.DataFrame(rows).set_index('Classifier')
    print(f'\n=== {label} ===')
    print(df_m.to_string())
    return df_m


print('Helper functions defined: run_loso(), compute_metrics(), summarise()')

## 6. Binary Classification: HC vs PD

The binary problem collapses PD-off and PD-on into a single PD class (label=1). This is the clinically most important question: *can we distinguish a Parkinson's patient from a healthy control using resting-state EEG alone?*

**Expected behaviour based on notebook 05 findings:**
- Large Cohen's d values (0.58–1.24) for theta features → classifier should detect the HC/PD separation
- HC vs PD-on has larger d than HC vs PD-off → PD-on sessions may be easier to classify correctly
- Balanced accuracy is the right metric here because HC=16 and PD=30 (1:1.9 ratio)

### 6.1. Primary Features — 7 FDR-Significant Theta Features

In [ ]:
print(f'Running Binary LOSO (HC vs PD) — Primary features ({len(sel_feat_cols)} theta)...')
bin_prim = run_loso(df, sel_feat_cols, 'binary_label', CLASSIFIERS)
metrics_bin_prim = summarise(bin_prim, n_classes=2,
                              label=f'Binary: HC vs PD  |  Primary ({len(sel_feat_cols)} theta features)')

### 6.2. Extended Features — All 45 ROI Features

Using all 45 features tests whether adding delta, alpha, beta, gamma, and APF features improves or hurts performance. With n=46, overfitting risk increases with more features — balanced accuracy may *drop* compared to the 7-feature primary set even though the training fit looks better.

In [ ]:
print(f'Running Binary LOSO (HC vs PD) — Extended features ({len(all_feat_cols)} total)...')
bin_ext = run_loso(df, all_feat_cols, 'binary_label', CLASSIFIERS)
metrics_bin_ext = summarise(bin_ext, n_classes=2,
                             label=f'Binary: HC vs PD  |  Extended ({len(all_feat_cols)} features)')

## 7. Three-Class Classification: HC vs PD-off vs PD-on

The 3-class problem is significantly harder than binary because it requires the model to distinguish between two PD states that differ primarily in medication level. From notebook 05:

- **HC vs PD-off boundary**: Large effect size (d up to 1.16) but high within-group variability in PD-off
- **HC vs PD-on boundary**: Largest effect sizes (d up to 1.24), statistically the cleanest separation
- **PD-off vs PD-on boundary**: Weakest separation (max d = 0.52) — the hardest boundary

For AUC-ROC we use the **macro-averaged One-vs-Rest** strategy, which computes a ROC curve for each class against all others and averages them. This is the standard for imbalanced multi-class evaluation.

### 7.1. Primary Features — 7 FDR-Significant Theta Features

In [ ]:
print(f'Running 3-Class LOSO (HC/PD-off/PD-on) — Primary features ({len(sel_feat_cols)} theta)...')
mc_prim = run_loso(df, sel_feat_cols, 'label', CLASSIFIERS)
metrics_mc_prim = summarise(mc_prim, n_classes=3,
                             label=f'3-Class: HC/PD-off/PD-on  |  Primary ({len(sel_feat_cols)} theta features)')

### 7.2. Extended Features — All 45 ROI Features

In [ ]:
print(f'Running 3-Class LOSO (HC/PD-off/PD-on) — Extended features ({len(all_feat_cols)} total)...')
mc_ext = run_loso(df, all_feat_cols, 'label', CLASSIFIERS)
metrics_mc_ext = summarise(mc_ext, n_classes=3,
                            label=f'3-Class: HC/PD-off/PD-on  |  Extended ({len(all_feat_cols)} features)')

## 8. Results Comparison

All four experiments (2 tasks × 2 feature sets) are merged into a single comparison table. **Balanced Accuracy** is the headline metric — it is honest about class imbalance and is the standard in clinical EEG classification papers.

In [ ]:
scenarios = {
    'Binary | Primary (7θ)':   (metrics_bin_prim, 2),
    'Binary | Extended (45)':  (metrics_bin_ext,  2),
    '3-Class | Primary (7θ)':  (metrics_mc_prim,  3),
    '3-Class | Extended (45)': (metrics_mc_ext,   3),
}

rows = []
for scenario, (mdf, _) in scenarios.items():
    for clf_name, row in mdf.iterrows():
        rows.append({'Scenario': scenario, 'Classifier': clf_name, **row.to_dict()})

comparison_df = pd.DataFrame(rows)
print('Full Results Comparison (primary metric: Balanced Accuracy):')
print(comparison_df.to_string(index=False))

# Save
out_path = os.path.join(PROC_DIR, 'classification_results.csv')
comparison_df.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')

## 9. Visualizations

### 9.1. Confusion Matrices

A confusion matrix shows the count of correct and incorrect predictions broken down by class. Each row is the **true** class; each column is the **predicted** class. Diagonal cells = correct predictions.

We show both binary and 3-class confusion matrices for the primary feature set (7 theta features), which is the scientifically validated input from notebook 05.

In [ ]:
label_map_bin = {0: 'HC', 1: 'PD'}
label_map_mc  = {0: 'HC', 1: 'PD-off', 2: 'PD-on'}

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.suptitle('Confusion Matrices — LOSO Cross-Validation (Primary: 7 Theta Features)',
             fontsize=14, fontweight='bold')

# Row 1: Binary
for col, (name, res) in enumerate(bin_prim.items()):
    ax = axes[0, col]
    y_true = np.array(res['y_true'])
    y_pred = np.array(res['y_pred'])
    cm     = confusion_matrix(y_true, y_pred)
    labels = [label_map_bin[c] for c in sorted(np.unique(y_true))]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels, yticklabels=labels, cbar=False, linewidths=0.5)
    ba = balanced_accuracy_score(y_true, y_pred)
    ax.set_title(f'{name}\nBinary | BalAcc = {ba:.2f}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

# Row 2: 3-class
for col, (name, res) in enumerate(mc_prim.items()):
    ax = axes[1, col]
    y_true = np.array(res['y_true'])
    y_pred = np.array(res['y_pred'])
    cm     = confusion_matrix(y_true, y_pred)
    labels = [label_map_mc[c] for c in sorted(np.unique(y_true))]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=ax,
                xticklabels=labels, yticklabels=labels, cbar=False, linewidths=0.5)
    ba = balanced_accuracy_score(y_true, y_pred)
    ax.set_title(f'{name}\n3-Class | BalAcc = {ba:.2f}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.tight_layout()
plt.show()

### 9.2. ROC Curves

The ROC (Receiver Operating Characteristic) curve plots the True Positive Rate (sensitivity) against the False Positive Rate (1 − specificity) across all decision thresholds. A perfect classifier sits at the top-left corner (TPR=1, FPR=0). AUC=0.5 is random chance.

- **Binary ROC**: Standard single curve — probability of predicting PD vs threshold
- **3-class ROC**: One curve per class (One-vs-Rest), then macro-averaged

This plot is particularly useful for clinical applications because it shows how the sensitivity/specificity trade-off changes as you move the decision threshold — important if the cost of a false negative (missed PD) differs from a false positive (wrongly diagnosed PD).

In [ ]:
# ── Binary ROC ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('ROC Curves — Binary Classification (HC vs PD)\nPrimary Features (7 theta)',
             fontsize=13, fontweight='bold')

for ax, (name, res) in zip(axes, bin_prim.items()):
    y_true = np.array(res['y_true'])
    y_prob = np.array(res['y_prob'])[:, 1]  # probability of PD class
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score   = auc(fpr, tpr)

    ax.plot(fpr, tpr, color='tomato', lw=2.5, label=f'AUC = {auc_score:.3f}')
    ax.fill_between(fpr, tpr, alpha=0.08, color='tomato')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.5)')
    ax.set_xlabel('False Positive Rate', fontsize=10)
    ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.set_title(name, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── 3-Class ROC (One-vs-Rest) ────────────────────────────────────────────────
class_labels  = [0, 1, 2]
class_names   = ['HC', 'PD-off', 'PD-on']
class_colors  = ['steelblue', 'tomato', 'darkorange']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('ROC Curves — 3-Class Classification (One-vs-Rest)\nPrimary Features (7 theta)',
             fontsize=13, fontweight='bold')

for ax, (name, res) in zip(axes, mc_prim.items()):
    y_true = np.array(res['y_true'])
    y_prob = np.array(res['y_prob'])
    y_bin  = label_binarize(y_true, classes=class_labels)

    auc_scores = []
    for i, (cn, cc) in enumerate(zip(class_names, class_colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        auc_i = auc(fpr, tpr)
        auc_scores.append(auc_i)
        ax.plot(fpr, tpr, color=cc, lw=2, label=f'{cn} (AUC={auc_i:.2f})')

    macro_auc = np.mean(auc_scores)
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate', fontsize=10)
    ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.set_title(f'{name}\nMacro AUC = {macro_auc:.3f}', fontweight='bold')
    ax.legend(loc='lower right', fontsize=8)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 9.3. Feature Importance (Random Forest)

Random Forest assigns an importance score to each feature based on how much it reduces impurity (Gini) across all splits in all trees. A higher score means the feature was more useful for making decisions.

Here we compute importance **averaged across all 31 LOSO folds** for the binary task. This gives a stable estimate that is not influenced by any single subject. Error bars show the standard deviation across folds.

This directly informs notebook 08 (SHAP interpretation): features with high importance here will be the primary targets for SHAP value analysis and scalp topomap visualisation.

In [ ]:
# Collect per-fold RF feature importances for both feature sets (binary task)
imp_prim_folds = []
imp_ext_folds  = []

for test_sub in subjects:
    train = df[df['subject'] != test_sub]
    test  = df[df['subject'] == test_sub]

    y_train = train['binary_label'].values

    # Primary
    sc = StandardScaler()
    X_tr = sc.fit_transform(train[sel_feat_cols])
    rf = clone(CLASSIFIERS['RF'])
    rf.fit(X_tr, y_train)
    imp_prim_folds.append(rf.feature_importances_)

    # Extended
    X_tr = sc.fit_transform(train[all_feat_cols])
    rf = clone(CLASSIFIERS['RF'])
    rf.fit(X_tr, y_train)
    imp_ext_folds.append(rf.feature_importances_)

imp_prim_mean = np.array(imp_prim_folds).mean(axis=0)
imp_prim_std  = np.array(imp_prim_folds).std(axis=0)
imp_ext_mean  = np.array(imp_ext_folds).mean(axis=0)
imp_ext_std   = np.array(imp_ext_folds).std(axis=0)

# ── Plot 1: Primary features ─────────────────────────────────────────────────
order = np.argsort(imp_prim_mean)[::-1]
names_ord = [sel_feat_cols[i] for i in order]
imp_ord   = imp_prim_mean[order]
std_ord   = imp_prim_std[order]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(len(names_ord)), imp_ord, yerr=std_ord, color='steelblue',
       alpha=0.8, edgecolor='k', linewidth=0.5, capsize=4, error_kw={'linewidth': 1})
ax.set_xticks(range(len(names_ord)))
ax.set_xticklabels(names_ord, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Mean Decrease in Impurity')
ax.set_title('RF Feature Importance — Binary (HC vs PD)\nPrimary features, averaged over 31 LOSO folds ± SD',
             fontweight='bold')
plt.tight_layout()
plt.show()

# ── Plot 2: Top 15 extended features ─────────────────────────────────────────
top_idx   = np.argsort(imp_ext_mean)[::-1][:15]
names_top = [all_feat_cols[i] for i in top_idx]
imp_top   = imp_ext_mean[top_idx]
std_top   = imp_ext_std[top_idx]

fig, ax = plt.subplots(figsize=(10, 4.5))
colors_top = ['tomato' if 'theta' in n else 'steelblue' for n in names_top]
ax.bar(range(len(names_top)), imp_top, yerr=std_top, color=colors_top,
       alpha=0.8, edgecolor='k', linewidth=0.5, capsize=4, error_kw={'linewidth': 1})
ax.set_xticks(range(len(names_top)))
ax.set_xticklabels(names_top, rotation=38, ha='right', fontsize=8.5)
ax.set_ylabel('Mean Decrease in Impurity')
ax.set_title('RF Feature Importance — Binary (HC vs PD)\nTop 15 from Extended (45) features  |  red = theta band',
             fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 15 extended features by RF importance:')
for i, (n, v) in enumerate(zip(names_top, imp_top)):
    print(f'  {i+1:2d}. {n:<28s}  {v:.4f} ± {std_top[i]:.4f}')

### 9.4. Per-Subject Accuracy

The LOSO framework gives us one prediction per session. This lets us ask: **which subjects are hardest to classify?**

Subjects with 0% accuracy are entirely misclassified — they may be outliers, have very few epochs, or represent edge cases (e.g., early-stage PD with minimal EEG changes, or HC subjects with elevated theta for non-disease reasons).

We plot per-subject accuracy for the binary task (SVM, primary features) since this is the cleanest comparison. Blue = HC subjects, red = PD subjects.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Per-Subject Accuracy — Binary Classification (HC vs PD)\nPrimary Features (7 theta)',
             fontsize=12, fontweight='bold')

for ax, (name, res) in zip(axes, bin_prim.items()):
    sub_df = pd.DataFrame({
        'subject': res['subject'],
        'y_true':  res['y_true'],
        'y_pred':  res['y_pred'],
    })
    sub_acc = (sub_df.groupby('subject')
               .apply(lambda x: (x['y_true'] == x['y_pred']).mean())
               .reset_index())
    sub_acc.columns = ['subject', 'accuracy']
    sub_acc['group']    = sub_acc['subject'].apply(lambda s: 'HC' if 'hc' in s else 'PD')
    sub_acc['n_epochs'] = sub_acc['subject'].map(df.set_index('subject')['n_epochs'])
    sub_acc = sub_acc.sort_values(['group', 'accuracy'], ascending=[True, False])

    colors = ['steelblue' if 'hc' in s else 'tomato' for s in sub_acc['subject']]
    bars = ax.barh(sub_acc['subject'], sub_acc['accuracy'],
                   color=colors, alpha=0.80, edgecolor='white', linewidth=0.4)
    ax.axvline(0.5, color='red', linestyle='--', linewidth=1.2, alpha=0.7, label='Chance')
    ax.axvline(1.0, color='gray', linestyle=':',  linewidth=0.8)
    ax.set_xlabel('Accuracy', fontsize=10)
    ax.set_title(name, fontweight='bold')
    ax.set_xlim([0, 1.12])
    ax.tick_params(axis='y', labelsize=7)

    # Annotate n_epochs for the worst subjects
    for _, row_s in sub_acc[sub_acc['accuracy'] < 0.6].iterrows():
        sub_idx = sub_acc['subject'].tolist().index(row_s['subject'])
        ax.text(row_s['accuracy'] + 0.02, sub_idx,
                f"n={int(row_s['n_epochs'])}", va='center', fontsize=6.5, color='gray')

axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

# Summary: hardest subjects
print('Subjects with accuracy < 0.6 (binary SVM, primary features):')
res_svm = bin_prim['SVM']
sub_df = pd.DataFrame({'subject': res_svm['subject'], 'y_true': res_svm['y_true'], 'y_pred': res_svm['y_pred']})
sub_acc_svm = (sub_df.groupby('subject')
               .apply(lambda x: (x['y_true'] == x['y_pred']).mean())
               .reset_index())
sub_acc_svm.columns = ['subject', 'accuracy']
sub_acc_svm['n_epochs'] = sub_acc_svm['subject'].map(df.set_index('subject')['n_epochs'])
sub_acc_svm['group'] = sub_acc_svm['subject'].apply(lambda s: 'HC' if 'hc' in s else 'PD')
hard = sub_acc_svm[sub_acc_svm['accuracy'] < 0.6].sort_values('accuracy')
if len(hard):
    print(hard[['subject', 'group', 'n_epochs', 'accuracy']].to_string(index=False))
else:
    print('  None — all subjects above chance level')

## 10. Conclusions

### 10.1. Classification Results

**Binary — HC vs PD (Primary 7θ features):**

| Classifier | Bal. Accuracy | F1 Macro | AUC-ROC |
|------------|--------------|----------|---------|
| **SVM** | **0.756** | 0.731 | 0.700 |
| RF | 0.683 | 0.692 | 0.658 |
| LDA | 0.667 | 0.673 | 0.621 |

SVM (RBF) achieves the best binary discrimination at **75.6% balanced accuracy** using only 7 theta features. This is well above chance (50%) and consistent with the large Cohen's d values found in notebook 05 (d = 0.58–1.24 across theta features). Chance-level in binary classification is 50%; a naive majority classifier that always predicts PD would reach only ~65% accuracy — balanced accuracy corrects for this.

**Binary — Extended (45) vs Primary (7θ):**  
Adding all 45 features *hurts* performance across all classifiers (LDA drops to 0.512, RF to 0.556, SVM to 0.662). This is a direct consequence of the small sample size (n=46): more features increase the model's degrees of freedom faster than they add discriminative signal, leading to overfitting within each LOSO fold. This validates the statistical feature selection performed in notebook 05 — the 7 FDR-significant theta features are both necessary and sufficient.

**3-class — HC vs PD-off vs PD-on (Primary 7θ features):**

| Classifier | Bal. Accuracy | F1 Macro | AUC-ROC |
|------------|--------------|----------|---------|
| **RF** | **0.588** | 0.577 | 0.694 |
| SVM | 0.567 | 0.554 | 0.583 |
| LDA | 0.476 | 0.470 | 0.625 |

Three-class performance is lower, with a chance level of 0.333. RF achieves 0.588 — a moderate but meaningful improvement over chance. LDA struggles here (0.476), likely because the PD-off vs PD-on boundary is non-linear and the equal-covariance assumption is violated. RF is the most robust classifier for 3-class, consistent with its ability to handle correlated features and non-linear boundaries.

The hardest classification boundary is PD-off vs PD-on — consistent with the notebook 05 finding that PD-off vs PD-on has max Cohen's d of only 0.52. The model likely learns HC correctly but confuses PD-off and PD-on with each other, which the confusion matrices confirm.

### 10.2. Why Primary Features Outperform Extended Features

This is a textbook example of the **bias-variance trade-off** at small n:
- 7 features → low-variance estimates; the model captures the dominant theta signal without memorising noise
- 45 features → high-variance estimates; each LOSO training fold has ~44 samples and 45 features, giving the model too many degrees of freedom\n- LDA is most affected (collapses to near-chance with 45 features) because it fits a full covariance matrix — impossible to estimate reliably at n≈44\n- RF is most robust to the extended set because its `max_features='sqrt'` and bootstrap sampling provide internal regularization

### 10.3. Model Comparison Summary

| Model | Best at | Why |
|-------|---------|-----|
| **SVM** | Binary (small feature set) | Margin maximization generalises best at n=46 with 7 features |
| **RF** | 3-class (either feature set) | Handles non-linear PD-off/PD-on boundary; most stable across feature set size |\n| **LDA** | Interpretability | Simplest model, good baseline, most sensitive to overfitting |

### 10.4. Caveats
- **Small sample size (n=46)**: Results are exploratory. LOSO balanced accuracy variance across folds is high (~±0.1 fold-to-fold). Do not interpret specific accuracy values as clinical performance estimates.
- **PD session inflation**: PD subjects contribute 2 sessions each. Binary training sets have 30 PD vs 16 HC sessions. `class_weight='balanced'` corrects for this during training, but the data structure inherently represents PD patterns more than HC.
- **Sub-pd6 and sub-pd16**: These subjects used EEGLAB-preprocessed ses-on data. Their LOSO folds may differ from others due to pipeline differences.
- **Theta-only features**: The 7 selected features are all theta band. The delta and APF features showed large but non-FDR-significant Cohen's d — they may contribute additional signal for deep learning (notebook 07) which does not require explicit feature selection.

### 10.5. Next Steps

| Notebook | Focus |
|----------|-------|
| **07** `07_deep_learning.ipynb` | EEGNet on raw epoch tensors — end-to-end learning without manual feature engineering |
| **08** `08_interpretation.ipynb` | SHAP values on RF (highest 3-class performance) + scalp topomaps to visualise *where* on the brain the classification signal originates |"
